# MBRL Curvature — Colab GPU launcher

Runs the GPU side of the project (Mode A: full loop; Mode B: train on shards
collected locally). Designed for Colab Pro session death: checkpoints push to
W&B every `checkpoint.every` updates and `checkpoint.resume=auto` picks up the
newest one on relaunch — just re-run all cells.

**Runtime → Change runtime type → A100** (L4/T4 fine for Pendulum-class runs).

In [10]:
# 1. GPU sanity
import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

2.11.0+cu128 | cuda: True | NVIDIA L4


In [ ]:
# 2. Get the code — idempotent and nesting-proof: clones if needed, then cds
# to wherever pyproject.toml actually lives (repos often nest the project,
# e.g. /content/mbrl/mbrl, and %cd on a re-run compounds the error).
import os, subprocess
REPO_URL = "https://github.com/BABYZ3US/MBRL.git"   # e.g. "https://github.com/you/mbrl-curvature.git"

os.chdir("/content")
if REPO_URL and not os.path.exists("/content/_repo"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/_repo"], check=True)

hits = subprocess.run(
    ["find", "/content", "-maxdepth", "4", "-name", "pyproject.toml",
     "-not", "-path", "*/.*"], capture_output=True, text=True).stdout.split()
assert hits, "no pyproject.toml under /content — clone or upload the project first"
PROJECT = os.path.dirname(sorted(hits, key=len)[0])
os.chdir(PROJECT)
print("project:", PROJECT)


/content/mbrl
ERROR: file:///content/mbrl does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
!pip -q install -e ".[mujoco]" 2>&1 | tail -1
!python -c "import mbrl, gymnasium; print('mbrl + gymnasium import ok')"


In [4]:
# 3. W&B login — works on any frontend.
import os
key = os.environ.get("WANDB_API_KEY")
if not key:
    try:                       # web Colab Secrets (key icon), if available
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        from getpass import getpass
        key = getpass("WANDB_API_KEY (from wandb.ai/authorize): ")
os.environ["WANDB_API_KEY"] = key
import wandb; wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pandelak (pandelak-boston-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
# 4a. PHASE 0 — regression gate (doses from docs/original_findings_report.md):
# recipe on HalfCheetah, 3 seeds sharing this GPU. Pass criterion ~ +98 +- 23.
!python scripts/parallel_runs.py --preset colab_recipe \
    --overrides env=halfcheetah model.latent_dim=17 --seeds 0 1 2 --jobs 3

python3: can't open file '/content/scripts/parallel_runs.py': [Errno 2] No such file or directory


In [ ]:
# 4b. Mode B — import locally-collected replay shards first (optional)
# import wandb
# art = wandb.Api().artifact("you/mbrl-curvature/replay-HalfCheetah-v5:latest")
# shard_dir = art.download()
# then pass +buffer.shards=$shard_dir to train.py (wire-up in train.py when needed)

In [ ]:
# 5. Join a W&B sweep (GPU agent). Local CPU agents can join the same sweep id.
# SWEEP_ID = "you/mbrl-curvature/abc123"
# !wandb agent $SWEEP_ID

In [ ]:
# 6. Keep-alive / disconnect drill: simulate a kill and verify resume works.
# !timeout 120 python scripts/train.py seed=0   # dies after 2 min
# !python scripts/train.py seed=0 checkpoint.resume=auto   # must continue, not restart